In [2]:
import torch
import numpy as np
import soundfile as sf
import gradio as gr
import time
import gc

from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import VitsModel, VitsTokenizer
from scipy.signal import resample_poly
from IPython.display import Audio, display
from google.colab import files
from silero_vad import load_silero_vad, get_speech_timestamps

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cuda
GPU: Tesla T4


In [17]:
import ctypes
import nvidia.cublas
import nvidia.cudnn
from pathlib import Path

cublas_path = Path(nvidia.cublas.__path__[0]) / "lib"
cudnn_path = Path(nvidia.cudnn.__path__[0]) / "lib"

ctypes.CDLL(
    str(cublas_path / "libcublasLt.so.12"),
    mode=ctypes.RTLD_GLOBAL
)

ctypes.CDLL(
    str(cublas_path / "libcublas.so.12"),
    mode=ctypes.RTLD_GLOBAL
)

ctypes.CDLL(
    str(cudnn_path / "libcudnn.so.9"),
    mode=ctypes.RTLD_GLOBAL
)

print("CUDA libraries loaded")

CUDA libraries loaded


In [3]:
stt_model = WhisperModel(
    "small",
    device=device,
    compute_type="int8_float16" if device == "cuda" else "int8"
)

print("Speech-to-Text загружена")

Speech-to-Text загружена


In [4]:
translation_id = "facebook/nllb-200-distilled-600M"

translation_tokenizer = AutoTokenizer.from_pretrained(
    translation_id
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

print("Text-to-Text загружена")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Text-to-Text загружена


In [5]:
tts_models = {}
tts_tokenizers = {}

for lang in ["rus", "eng"]:
    model_id = f"facebook/mms-tts-{lang}"

    tts_tokenizers[lang] = VitsTokenizer.from_pretrained(model_id)

    tts_models[lang] = VitsModel.from_pretrained(
        model_id
    ).to(device).eval()

    print(f"Text-to-Speech ({lang}) загружена")

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Text-to-Speech (eng) загружена


In [6]:
vad_model = load_silero_vad()

print("Silero VAD загружена")

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Silero VAD загружена


In [7]:
LANGUAGES = {
    "Русский": {
        "stt": "ru",
        "translation": "rus_Cyrl",
        "tts": "rus"
    },
    "Английский": {
        "stt": "en",
        "translation": "eng_Latn",
        "tts": "eng"
    }
}

In [14]:
def speech_to_text(audio, source_lang):

    if not isinstance(audio, np.ndarray):
        import subprocess

        process = subprocess.run(
            [
                "ffmpeg",
                "-nostdin",
                "-i", str(audio),
                "-f", "f32le",
                "-ac", "1",
                "-ar", "16000",
                "pipe:1"
            ],
            capture_output=True,
            check=True
        )

        audio = np.frombuffer(
            process.stdout,
            dtype=np.float32
        ).copy()

    segments, info = stt_model.transcribe(
        audio,
        language=LANGUAGES[source_lang]["stt"],
        beam_size=3,
        vad_filter=True,
        condition_on_previous_text=False
    )

    text = " ".join(segment.text.strip() for segment in segments)

    return text.strip()

In [9]:
@torch.inference_mode()
def translate_text(text, source_lang, target_lang):

    translation_tokenizer.src_lang = LANGUAGES[source_lang]["translation"]

    inputs = translation_tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    tokens = translation_model.generate(
        **inputs,
        forced_bos_token_id=translation_tokenizer.convert_tokens_to_ids(
            LANGUAGES[target_lang]["translation"]
        ),
        max_new_tokens=256,
        num_beams=3
    )

    translated_text = translation_tokenizer.batch_decode(
        tokens,
        skip_special_tokens=True
    )[0]

    return translated_text

In [10]:
@torch.inference_mode()
def text_to_speech(text, target_lang):

    lang = LANGUAGES[target_lang]["tts"]

    tokenizer = tts_tokenizers[lang]
    model = tts_models[lang]

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    waveform = model(**inputs).waveform

    audio = waveform.squeeze().cpu().numpy()
    sample_rate = model.config.sampling_rate

    return sample_rate, audio

In [11]:
def voice_translate(audio, source_lang, target_lang):

    start = time.perf_counter()

    original_text = speech_to_text(audio, source_lang)
    t1 = time.perf_counter()

    if not original_text:
        raise ValueError("Речь не обнаружена")

    translated_text = translate_text(
        original_text,
        source_lang,
        target_lang
    )
    t2 = time.perf_counter()

    sample_rate, generated_audio = text_to_speech(
        translated_text,
        target_lang
    )
    t3 = time.perf_counter()

    print("Исходный текст:", original_text)
    print("Перевод:", translated_text)

    print(f"\nSpeech-to-Text: {t1 - start:.2f} сек.")
    print(f"Text-to-Text: {t2 - t1:.2f} сек.")
    print(f"Text-to-Speech: {t3 - t2:.2f} сек.")
    print(f"Общее время: {t3 - start:.2f} сек.")

    return original_text, translated_text, sample_rate, generated_audio

In [12]:
uploaded = files.upload()

audio_path = next(iter(uploaded))

display(Audio(filename=audio_path))

Saving testaudio.m4a to testaudio.m4a


In [18]:
original_text, translated_text, sr, audio = voice_translate(
    audio_path,
    source_lang="Русский",
    target_lang="Английский"
)

display(Audio(audio, rate=sr))

sf.write("translated_audio.wav", audio, sr)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Исходный текст: Привет, сегодня я тестирую голосовой переводчик.
Перевод: Hey, I'm testing a voice translator today.

Speech-to-Text: 1.87 сек.
Text-to-Text: 2.95 сек.
Text-to-Speech: 2.41 сек.
Общее время: 7.22 сек.


In [20]:
SAMPLE_RATE = 16000
CHUNK_DURATION = 0.5
SILENCE_DURATION = 0.7
MIN_SPEECH_DURATION = 0.3
MAX_PHRASE_DURATION = 12

VAD_THRESHOLD = 0.5

In [21]:
from math import gcd

def prepare_audio(audio_chunk):

    sr, audio = audio_chunk

    if np.issubdtype(audio.dtype, np.integer):
        audio = audio.astype(np.float32) / max(
            abs(np.iinfo(audio.dtype).min),
            np.iinfo(audio.dtype).max
        )
    else:
        audio = audio.astype(np.float32)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != SAMPLE_RATE:

        divisor = gcd(sr, SAMPLE_RATE)

        audio = resample_poly(
            audio,
            SAMPLE_RATE // divisor,
            sr // divisor
        )

    return audio.astype(np.float32)

In [22]:
def detect_speech(audio):

    timestamps = get_speech_timestamps(
        torch.from_numpy(audio),
        vad_model,
        sampling_rate=SAMPLE_RATE,
        threshold=VAD_THRESHOLD,
        min_speech_duration_ms=int(
            MIN_SPEECH_DURATION * 1000
        ),
        min_silence_duration_ms=int(
            SILENCE_DURATION * 1000
        )
    )

    return timestamps

In [23]:
def create_state():

    return {
        "buffer": np.array([], dtype=np.float32),
        "original_history": "",
        "translated_history": "",
        "phrase_count": 0
    }

In [24]:
def realtime_translate(
    audio_chunk,
    source_lang,
    target_lang,
    state
):

    if state is None:
        state = create_state()

    if audio_chunk is None:
        return (
            state,
            gr.skip(),
            gr.skip(),
            gr.skip(),
            "Ожидание речи..."
        )

    chunk = prepare_audio(audio_chunk)

    state["buffer"] = np.concatenate([
        state["buffer"],
        chunk
    ])

    buffer = state["buffer"]

    timestamps = detect_speech(buffer)

    if not timestamps:

        state["buffer"] = buffer[-SAMPLE_RATE:]

        return (
            state,
            gr.skip(),
            gr.skip(),
            gr.skip(),
            "Ожидание речи..."
        )

    last_speech_end = timestamps[-1]["end"]

    silence = (
        len(buffer) - last_speech_end
    ) / SAMPLE_RATE

    duration = len(buffer) / SAMPLE_RATE

    phrase_finished = (
        silence >= SILENCE_DURATION
    )

    max_duration_reached = (
        duration >= MAX_PHRASE_DURATION
    )

    if not phrase_finished and not max_duration_reached:

        return (
            state,
            gr.skip(),
            gr.skip(),
            gr.skip(),
            "Слушаю..."
        )

    phrase = buffer.copy()

    state["buffer"] = np.array(
        [],
        dtype=np.float32
    )

    start = time.perf_counter()

    original = speech_to_text(
        phrase,
        source_lang
    )

    if not original:

        return (
            state,
            gr.skip(),
            gr.skip(),
            gr.skip(),
            "Речь не распознана"
        )

    translated = translate_text(
        original,
        source_lang,
        target_lang
    )

    sr, generated_audio = text_to_speech(
        translated,
        target_lang
    )

    latency = time.perf_counter() - start

    state["phrase_count"] += 1

    state["original_history"] += original + "\n"
    state["translated_history"] += translated + "\n"

    status = (
        f'Фраз обработано: {state["phrase_count"]} | '
        f'Время обработки: {latency:.2f} сек.'
    )

    return (
        state,
        state["original_history"],
        state["translated_history"],
        (sr, generated_audio),
        status
    )

In [25]:
if "demo" in globals():
    demo.close()

with gr.Blocks() as realtime_demo:

    gr.Markdown("# Real-Time AI Voice Translator")

    gr.Markdown(
        "Faster Whisper + NLLB-200 + MMS-TTS + Silero VAD"
    )

    state = gr.State(value=create_state)

    with gr.Row():

        source_lang = gr.Dropdown(
            choices=list(LANGUAGES),
            value="Русский",
            label="Исходный язык"
        )

        target_lang = gr.Dropdown(
            choices=list(LANGUAGES),
            value="Английский",
            label="Язык перевода"
        )

    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="Микрофон"
    )

    status = gr.Textbox(
        label="Статус",
        value="Ожидание речи...",
        interactive=False
    )

    with gr.Row():

        original_output = gr.Textbox(
            label="Распознанный текст",
            lines=8,
            interactive=False
        )

        translation_output = gr.Textbox(
            label="Переведённый текст",
            lines=8,
            interactive=False
        )

    audio_output = gr.Audio(
        label="Голосовой перевод",
        autoplay=True,
        interactive=False
    )

    clear_button = gr.Button("Очистить историю")

    microphone.stream(
        fn=realtime_translate,
        inputs=[
            microphone,
            source_lang,
            target_lang,
            state
        ],
        outputs=[
            state,
            original_output,
            translation_output,
            audio_output,
            status
        ],
        stream_every=CHUNK_DURATION,
        time_limit=3600,
        concurrency_limit=1,
        show_progress="hidden"
    )

    clear_button.click(
        fn=lambda: (
            create_state(),
            "",
            "",
            None,
            "Ожидание речи..."
        ),
        outputs=[
            state,
            original_output,
            translation_output,
            audio_output,
            status
        ]
    )

Closing server running on port: 7860


In [26]:
realtime_demo.queue().launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c95e80e3adccbd89f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[tr

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9da0ada170e92aeaf3.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://c95e80e3adccbd89f5.gradio.live
